# Evaluacion con Conjunto Expandido (6,600 Registros)

Este cuaderno realiza la evaluacion utilizando el conjunto ampliado con 500 nuevos registros de refuerzo cotidiano.


In [ ]:
# 1. Clonar el repositorio si estamos ejecutando en Colab
import os
if not os.path.exists('satire_detector_api'):
    !git clone https://github.com/luisknight24/DeteccionSatira.git
    %cd DeteccionSatira
else:
    print("Ya estás dentro de la carpeta del proyecto.")

In [ ]:
# 2. Instalar dependencias necesarias
!pip install -r requirements.txt
!python -m spacy download es_core_news_sm

In [ ]:
# 3. Configurar entorno Django para poder utilizar utilidades de preprocesamiento
import sys
sys.path.append(os.path.abspath('satire_detector_api'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'satire_detector_api.settings')

import django
django.setup()

from detector.utils.text_processor import TextProcessor
from detector.utils.bert_classifier import BertClassifier

In [ ]:
# 4. Cargar y verificar el dataset
import pandas as pd
import numpy as np

ruta_original = "documentos_origen/Titulacion1/DatasetsFinales/df_train2_featselect2.jsonl"
df = pd.read_json(ruta_original, orient='records', lines=True)
print(f"El dataset contiene originalmente {len(df)} registros.")

# Comprobar si ya contiene los ejemplos de refuerzo
df['es_refuerzo'] = df['id'].astype(str).str.startswith('refuerzo_')
num_refuerzo = df['es_refuerzo'].sum()

if num_refuerzo == 0:
    print("El dataset no contiene los ejemplos de refuerzo. Agregando dinámicamente desde el script...")
    from satire_detector_api.reforzar_modelo import ejemplos_refuerzo, selected_features
    
    processor = TextProcessor()
    nuevos_registros = []
    for idx, ejemplo in enumerate(ejemplos_refuerzo):
        text = ejemplo["text"]
        label = ejemplo["label"]
        processed_text = processor.preprocess_text(text)
        features_dict = processor.calculate_features(text)
        
        registro = {
            "id": f"refuerzo_{idx}",
            "transcription": text,
            "transcription_processed": processed_text,
            "label": label
        }
        for feat in selected_features:
            registro[feat] = features_dict.get(feat, 0)
        nuevos_registros.append(registro)
    df_nuevos = pd.DataFrame(nuevos_registros)
    df = pd.concat([df, df_nuevos], ignore_index=True)
    df['es_refuerzo'] = df['id'].astype(str).str.startswith('refuerzo_')
    print(f"Dataset expandido exitosamente a {len(df)} registros (6,100 esperados).")
else:
    print(f"El dataset ya contiene los {num_refuerzo} ejemplos de refuerzo (Total: {len(df)} registros).")

In [ ]:
# 5. Partición Train/Test (80/20) y Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from satire_detector_api.reforzar_modelo import selected_features

X_text = df['transcription_processed'].fillna("").values
X_manual = df[selected_features].values
y = df['label'].values
es_refuerzo = df['es_refuerzo'].values

indices = np.arange(len(df))
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    indices, y, test_size=0.2, random_state=42, stratify=y
)

test_es_refuerzo = es_refuerzo[X_test_idx]
print(f"Entrenamiento: {len(X_train_idx)} muestras")
print(f"Prueba: {len(X_test_idx)} muestras")
print(f"  └─ Muestras de prueba originales (In-Domain): {np.sum(~test_es_refuerzo)}")
print(f"  └─ Muestras de prueba de refuerzo (Out-of-Domain): {np.sum(test_es_refuerzo)}")

# Extracción TF-IDF
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
tfidf_train = vectorizer.fit_transform(X_text[X_train_idx]).toarray()
tfidf_test = vectorizer.transform(X_text[X_test_idx]).toarray()

# Combinación y Normalización
X_train_combined = np.concatenate([tfidf_train, X_manual[X_train_idx]], axis=1)
X_test_combined = np.concatenate([tfidf_test, X_manual[X_test_idx]], axis=1)

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_combined)
X_test_imputed = imputer.transform(X_test_combined)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# SMOTE Balance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
# 6. Entrenar y Evaluar Modelos Tradicionales
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

SVC_model = SVC(kernel='linear', probability=True, random_state=42)
RF_model = RandomForestClassifier(n_estimators=100, random_state=42)
MLP_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
XGB_model = XGBClassifier(n_estimators=100, random_state=42)
VC_model = VotingClassifier(
    estimators=[('RF', RF_model), ('SVM', SVC_model), ('MLP', MLP_model), ('XGB', XGB_model)],
    voting='soft'
)

models = {
    'SVM': SVC_model,
    'Random Forest': RF_model,
    'MLP': MLP_model,
    'XGBoost': XGB_model,
    'Voting Classifier': VC_model
}

trad_results = {}
for name, model in models.items():
    print(f"Entrenando {name}...")
    model.fit(X_train_balanced, y_train_balanced)
    y_pred = model.predict(X_test_scaled)
    
    # Métricas
    acc_glob = accuracy_score(y_test, y_pred)
    f1_glob = f1_score(y_test, y_pred, average='macro')
    
    # Segmentadas
    acc_in = accuracy_score(y_test[~test_es_refuerzo], y_pred[~test_es_refuerzo])
    acc_out = accuracy_score(y_test[test_es_refuerzo], y_pred[test_es_refuerzo])
    
    trad_results[name] = {
        "Global_Acc": acc_glob,
        "Global_F1": f1_glob,
        "InDomain_Acc": acc_in,
        "OutDomain_Acc": acc_out
    }
    
pd.DataFrame(trad_results).T

In [ ]:
# 7. Evaluar el Modelo Híbrido Real (BETO)
import torch
from transformers import BertTokenizerFast

model_path = "satire_detector_api/static/best_model_spanish_loss.pt"
tokenizer_path = "satire_detector_api/static/model_files/tokenizer_files"

if os.path.exists(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Cargando pesos del modelo en: {device}")
    
    tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path)
    
    # Registrar la clase en __main__ para evitar fallos de pickle
    import __main__
    __main__.BertClassifier = BertClassifier
    hybrid_model = torch.load(model_path, map_location=device, weights_only=False)
    hybrid_model.to(device)
    hybrid_model.eval()
    
    # Inferencia en lotes para mayor rapidez
    all_preds = []
    batch_size = 32
    texts_test = X_text[X_test_idx]
    
    for i in range(0, len(texts_test), batch_size):
        batch_texts = texts_test[i:i+batch_size]
        batch_extra = X_test_scaled[i:i+batch_size]
        
        encodings = tokenizer(
            batch_texts.tolist(), 
            truncation=True, 
            padding=True, 
            max_length=64, 
            return_tensors="pt"
        ).to(device)
        
        extra_tensor = torch.tensor(batch_extra, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            outputs = hybrid_model(encodings['input_ids'], encodings['attention_mask'], extra_tensor)
            probs = torch.exp(outputs)
            preds = (probs[:, 1] >= 0.55).cpu().numpy().astype(int)
            all_preds.extend(preds)
            
    all_preds = np.array(all_preds)
    
    # Calcular métricas finales de BETO
    acc_glob = accuracy_score(y_test, all_preds)
    f1_glob = f1_score(y_test, all_preds, average='macro')
    acc_in = accuracy_score(y_test[~test_es_refuerzo], all_preds[~test_es_refuerzo])
    acc_out = accuracy_score(y_test[test_es_refuerzo], all_preds[test_es_refuerzo])
    
    print("\n=== RESULTADOS FINALES DE BETO HÍBRIDO ===")
    print(f"Accuracy Global:         {acc_glob:.2%}")
    print(f"F1-Score Global (Macro): {f1_glob:.2%}")
    print(f"Accuracy In-Domain:      {acc_in:.2%}")
    print(f"Accuracy Out-of-Domain:  {acc_out:.2%}")
else:
    print("No se encontraron los pesos del modelo preentrenado. Asegúrate de subirlos a tu Drive o static/.")

## Visualización de Matrices de Confusión

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def graficar_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
                xticklabels=['Neutro', 'Sátira'], yticklabels=['Neutro', 'Sátira'])
    plt.title(title)
    plt.xlabel('Predicho')
    plt.ylabel('Verdadero')
    plt.show()

# Graficar para el Voting Classifier
graficar_confusion(y_test, models['Voting Classifier'].predict(X_test_scaled), "Voting Classifier - Test Completo")

# Graficar para BETO si está disponible
if 'all_preds' in locals():
    graficar_confusion(y_test, all_preds, "BETO Híbrido - Test Completo")